In [7]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
import polars
from skforecast.datasets import fetch_dataset
import sys

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
#plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import xgboost as xgb
import skforecast
import sklearn
from xgboost import XGBRegressor
from sklearn.feature_selection import RFECV
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.model_selection import bayesian_search_forecaster
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import select_features
import shap

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

from sklearn.preprocessing import LabelEncoder

Import dataframes

In [8]:
# xgdf = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/Aggregated_data_20241106.parquet', engine='pyarrow')
xgdf_train = pd.read_parquet('/Users/Georgi/Dropbox/Data and AI/Applications/Group Project Datasets/train_df_20241116.parquet', engine='pyarrow')
xgdf_test = pd.read_parquet('/Users/Georgi/Dropbox/Data and AI/Applications/Group Project Datasets/test_df_20241116.parquet', engine='pyarrow')
xgdf_val = pd.read_parquet('/Users/Georgi/Dropbox/Data and AI/Applications/Group Project Datasets/val_df_20241116.parquet', engine='pyarrow')

xgdf = pd.concat([xgdf_train, xgdf_test, xgdf_val], axis=0, ignore_index=True)

xgdf.info()
xgdf.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8466612 entries, 0 to 8466611
Data columns (total 17 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   store_nbr               uint8         
 1   item_nbr                uint32        
 2   date                    datetime64[ns]
 3   onpromotion             int64         
 4   holiday_local_count     int8          
 5   holiday_national_count  int8          
 6   holiday_regional_count  int8          
 7   store_type              category      
 8   store_cluster           uint8         
 9   item_family             category      
 10  item_class              uint16        
 11  perishable              uint8         
 12  store_status            int8          
 13  item_status             int8          
 14  year                    int16         
 15  week_number_cum         int16         
 16  unit_sales              float32       
dtypes: category(2), datetime64[ns](1), float32(1),

,store_nbr,item_nbr,date,onpromotion,holiday_local_count,holiday_national_count,holiday_regional_count,store_type,store_cluster,item_family,item_class,perishable,store_status,item_status,year,week_number_cum,unit_sales
0,1,103520,2013-01-02,0,0,1,0,D,13,GROCERY I,1028,0,0,1,2013,1,7.500000
1,1,103520,2013-01-07,0,0,1,0,D,13,GROCERY I,1028,0,0,1,2013,2,17.500000
2,1,103520,2013-01-14,0,0,0,0,D,13,GROCERY I,1028,0,0,1,2013,3,14.300000
3,1,103520,2013-01-21,0,0,0,0,D,13,GROCERY I,1028,0,0,1,2013,4,23.000000
4,1,103520,2013-01-28,0,0,0,0,D,13,GROCERY I,1028,0,0,1,2013,5,14.452381


Create extra features + One Hot Encoding for store type and item family

In [9]:
#Data transformations
xgdf['week_number'] = xgdf['date'].dt.isocalendar().week
xgdf['month'] = xgdf['date'].dt.month
xgdf['year'] = xgdf['date'].dt.year

xgdf['unique_id'] = xgdf['store_nbr'].astype(str) + "_" + xgdf['item_nbr'].astype(str)

# Encode `store_id`
le = LabelEncoder()
xgdf['unique_id'] = le.fit_transform(xgdf['unique_id'])

xgdf = pd.get_dummies(xgdf, columns=['store_type'])
xgdf = pd.get_dummies(xgdf, columns=['item_family'])

xgdf = xgdf.loc[:, ~(xgdf == False).all()]

xgdf

,store_nbr,item_nbr,date,onpromotion,holiday_local_count,holiday_national_count,holiday_regional_count,store_cluster,item_class,perishable,...,month,unique_id,store_type_A,store_type_B,store_type_C,store_type_D,item_family_BREAD/BAKERY,item_family_DAIRY,item_family_GROCERY I,item_family_POULTRY
0,1,103520,2013-01-02,0,0,1,0,13,1028,0,...,1,6671,False,False,False,True,False,False,True,False
1,1,103520,2013-01-07,0,0,1,0,13,1028,0,...,1,6671,False,False,False,True,False,False,True,False
2,1,103520,2013-01-14,0,0,0,0,13,1028,0,...,1,6671,False,False,False,True,False,False,True,False
3,1,103520,2013-01-21,0,0,0,0,13,1028,0,...,1,6671,False,False,False,True,False,False,True,False
4,1,103520,2013-01-28,0,0,0,0,13,1028,0,...,1,6671,False,False,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8466607,54,2089339,2017-01-16,0,0,0,0,3,1006,0,...,1,30458,False,False,True,False,False,False,True,False
8466608,54,2089339,2017-01-23,0,0,0,0,3,1006,0,...,1,30458,False,False,True,False,False,False,True,False
8466609,54,2089339,2017-01-30,0,0,0,0,3,1006,0,...,1,30458,False,False,True,False,False,False,True,False
8466610,54,2089339,2017-02-06,0,0,0,0,3,1006,0,...,2,30458,False,False,True,False,False,False,True,False


In [10]:
def add_lagged_features(df, lag_shift=2, rolling_mean_window=3):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["unique_id", "week_number_cum"])

    # Lag by the specified number of weeks
    df["y_naive"] = df["unit_sales"].shift(lag_shift)

    # Rolling mean (t-2, t-3, t-4)
    df["y_mean"] = (
        df["unit_sales"].shift(lag_shift).rolling(window=rolling_mean_window).mean()
    )

    # Group by (item_nbr, store_nbr) and apply the lagging and rolling features
    df = (
        df.groupby(["unique_id"], group_keys=False)
        .apply(lambda x: x)
        .reset_index(drop=True)
    )

    return df

In [11]:
xgdf = add_lagged_features(xgdf, lag_shift=2, rolling_mean_window=3)

xgdf[['store_nbr', 'item_nbr', 'week_number_cum', 'unit_sales', 'y_naive', 'y_mean']].head(20)


/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_72866/3258991551.py:17: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,store_nbr,item_nbr,week_number_cum,unit_sales,y_naive,y_mean
0,10,1001305,1,0.0,NaN,NaN
1,10,1001305,2,0.0,NaN,NaN
2,10,1001305,3,0.0,0.0,NaN
3,10,1001305,4,0.0,0.0,NaN
4,10,1001305,5,0.0,0.0,0.0
5,10,1001305,6,0.0,0.0,0.0
6,10,1001305,7,0.0,0.0,0.0
7,10,1001305,8,0.0,0.0,0.0
8,10,1001305,9,0.0,0.0,0.0
9,10,1001305,10,0.0,0.0,0.0


Create 52 lags

In [12]:
#Sort timeseries by unique id and date
xgdf=xgdf.sort_values(by=['unique_id','date'])

for i in range(1, 53):
    # Create the new column dynamically
    col_name = f'last{7 * i}Value'
    xgdf[col_name] = xgdf.groupby('unique_id')['unit_sales'].shift(i)

xgdf = xgdf.dropna()

xgdf.head()

,store_nbr,item_nbr,date,onpromotion,holiday_local_count,holiday_national_count,holiday_regional_count,store_cluster,item_class,perishable,...,last301Value,last308Value,last315Value,last322Value,last329Value,last336Value,last343Value,last350Value,last357Value,last364Value
52,10,1001305,2013-12-30,0,0,2,0,15,1016,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53,10,1001305,2014-01-06,0,0,0,0,15,1016,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,10,1001305,2014-01-13,0,0,0,0,15,1016,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55,10,1001305,2014-01-20,0,0,0,0,15,1016,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,10,1001305,2014-01-27,0,0,0,0,15,1016,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Normalize

In [13]:
def normalize(series):
    # Ensure the series is numeric (convert to float)
    series = series.astype(float)  # Convert to float directly
    min_val = series.min()
    max_val = series.max()
    return (series - min_val) / (max_val - min_val)

col_to_normalize = ['week_number', 'year', 'month']

n_xgdf = xgdf.copy() 

for column in col_to_normalize:
    n_xgdf[column] = normalize(n_xgdf[column])

Train test validation split

In [24]:
def train_val_test_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()
    min_week = df["week_number_cum"].min()

    # Calculate start and end weeks for validation and test sets
    test_week_start = max_week - window_length + 1

    val_week_start = max_week - 2 * window_length + 1

    val_week_end = test_week_start - 1

    train_week_end = val_week_start - 1

    train_week_start = max_week - 4 * window_length + 1
    # train_week_start = min_week

    # Train data: All data before the start of the validation period
    df_train = df[
        (df["week_number_cum"] >= train_week_start)
        & (df["week_number_cum"] <= train_week_end)
    ]
    # df_train = df[
    #     (df["week_number_cum"] >= train_week_start)
    #     & (df["week_number_cum"] <= val_week_end)
    # ]

    # Val data: From `val_week_start` to `val_week_end`
    df_test = df[
         (df["week_number_cum"] >= val_week_start)
         & (df["week_number_cum"] <= val_week_end)
     ]
    # df_test = df[
    #     (df["week_number_cum"] >= test_week_start) & (df["week_number_cum"] <= max_week)
    # ]


    # Test data: From `test_week_start` to `max_week`
    df_validate = df[
        (df["week_number_cum"] >= test_week_start) & (df["week_number_cum"] <= max_week)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")
        print(f"Size of {round(sys.getsizeof(split)/1024/1024/1024, 2)} GB.")

    # Print information about the splits
    print_split_info("Train", df_train)
    print_split_info("Test", df_test)
    print_split_info("Validate", df_validate)

    return df_train, df_test, df_validate

In [25]:
n_xgdf['date'] = pd.to_datetime(n_xgdf['date'])
df_train, df_test, df_validate = train_val_test_split(n_xgdf, window_length=26)

# 2. Sort by 'date'
df_train = df_train.sort_values(by='week_number_cum')
df_test = df_test.sort_values(by='week_number_cum')
df_validate = df_validate.sort_values(by='week_number_cum')

# 3. Drop 'unit_sales' and 'date' columns for X_train and X_test
X_train = df_train.drop(columns=['unit_sales', 'date', 'last7Value', 'y_naive', 'y_mean'])
X_test = df_test.drop(columns=['unit_sales', 'date', 'last7Value', 'y_naive', 'y_mean'])

# 4. Rename 'unit_sales' to 'Label' for y_train and y_test
y_train = df_train[['unit_sales']].rename(columns={'unit_sales': 'Label'})
y_test = df_test[['unit_sales']].rename(columns={'unit_sales': 'Label'})


Train set: shape: (1819272, 80)
Train Min Week: 139
Train Max Week: 190
Train number of weeks: 52
Number of stores: 42
Number of items: 833
Size of 0.52 GB.

Test set: shape: (909636, 80)
Test Min Week: 191
Test Max Week: 216
Test number of weeks: 26
Number of stores: 42
Number of items: 833
Size of 0.26 GB.

Validate set: shape: (909636, 80)
Validate Min Week: 217
Validate Max Week: 242
Validate number of weeks: 26
Number of stores: 42
Number of items: 833
Size of 0.26 GB.


In [28]:
X_train.head()

,store_nbr,item_nbr,onpromotion,holiday_local_count,holiday_national_count,holiday_regional_count,store_cluster,item_class,perishable,store_status,...,last301Value,last308Value,last315Value,last322Value,last329Value,last336Value,last343Value,last350Value,last357Value,last364Value
1614520,1,103520,0,0,0,0,13,1028,0,0,...,34.428570,20.000000,21.366667,18.428572,25.0,16.000000,17.4,17.642857,22.0,14.0
4880068,40,1239955,1,0,0,0,3,2174,1,0,...,267.000000,174.000000,211.000000,157.000000,98.0,152.000000,179.0,164.000000,206.0,215.0
1268944,16,1418844,0,0,0,0,3,1004,0,0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
2316,10,1037845,0,0,0,0,15,1022,0,0,...,42.000000,34.000000,43.000000,56.000000,65.0,35.142857,45.0,49.000000,83.0,52.0
3567460,33,409904,0,0,0,0,3,1034,0,0,...,60.714287,48.428574,46.000000,45.000000,59.0,39.714287,41.0,41.000000,47.0,58.0


Train XGB model

In [22]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred[y_pred < 0] = 0

# a = y_test.rename(columns={'Label': 'y'})
# b=pd.DataFrame(y_pred, columns=['y_p'])

In [26]:
print(type(y_pred))  # Type of y_pred
print(y_pred.shape)  # Shape of y_pred


<class 'numpy.ndarray'>
(909636,)


Create dataframe which contains predictions and metrics (bias and accuracy)

In [ ]:
# Create a copy of the test dataset to avoid modifying the original dataset
df_pred = df_test.copy()

# Retain only the specified columns in the new DataFrame
df_pred = df_pred[['store_nbr', 'item_nbr', 'unit_sales', 'week_number_cum', 'date', 'unique_id', 
                   'y_naive', 'y_mean', 'store_cluster', 'item_class', 'perishable', 
                   'store_type_A', 'store_type_B', 'store_type_C', 'store_type_D', 
                   'item_family_BREAD/BAKERY', 'item_family_DAIRY', 'item_family_GROCERY I', 'item_family_POULTRY']]

# Undo One Hot Encoding store_type and item_family
df_pred['store_type'] = df_pred.filter(like='store_type_').idxmax(axis=1)
df_pred['store_type'] = df_pred['store_type'].str.replace('store_type_', '')
df_pred = df_pred.drop(columns=xgdf.filter(like='store_type_').columns)

df_pred['item_family'] = df_pred.filter(like='item_family_').idxmax(axis=1)
df_pred['item_family'] = df_pred['item_family'].str.replace('item_family_', '')
df_pred = df_pred.drop(columns=xgdf.filter(like='item_family_').columns)

# Add a new column 'y_xgb' to store the predictions from the XGBoost model
df_pred['y_xgb'] = y_pred

# Calculate the bias (error) for XGBoost predictions and naive
df_pred['bias_xgb'] = df_pred['y_xgb'] - df_pred['unit_sales']
df_pred['bias_naive'] = df_pred['y_naive'] - df_pred['unit_sales']

# Calculate accuracy for XGBoost predictions, if unit_sales = 0 don't calculate
df_pred['acc_xgb'] = np.where(
    (df_pred['unit_sales'] == 0) & (df_pred['y_xgb'] != 0),
    np.nan,
    (1 - np.abs(df_pred['bias_xgb']) / df_pred['unit_sales']) * 100
)

# Calculate accuracy for naive predictions using the same logic as above
df_pred['acc_naive'] = np.where(
    (df_pred['unit_sales'] == 0) & (df_pred['y_naive'] != 0),
    np.nan,
    (1 - np.abs(df_pred['bias_naive']) / df_pred['unit_sales']) * 100
)

# Calculate effect of bias. Overprediction times 2/3 and underprediction times -1/3
df_pred['adj_bias_xgb'] = (df_pred['bias_xgb'] * 2 / 3).where(df_pred['bias_xgb'] >= 0, df_pred['bias_xgb'] * -1 / 3)
df_pred['adj_bias_naive'] = (df_pred['bias_naive'] * 2 / 3).where(df_pred['bias_naive'] >= 0, df_pred['bias_naive'] * -1 / 3)


# Sort the DataFrame by 'unique_id' and 'date' for better organization
df_pred = df_pred.sort_values(by=['unique_id', 'date'])

# Display the resulting DataFrame
df_pred

In [ ]:
drop_columns = ['bias_xgb', 'bias_naive', 'acc_xgb', 'acc_naive', 'adj_bias_xgb', 'adj_bias_naive']

df_pred = df_pred.drop(columns=drop_columns)

df_pred.to_parquet("C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Applications/Group4B/data/processed/df_pred20241120.parquet", engine='pyarrow', index=False)

Define functions for calculation metrics, plots and histograms

In [ ]:
def calculate_metrics(df):
    # Calculate means for unit sales and predictions
    mean_a = np.mean(df['unit_sales'])
    mean_p = np.mean(df['y_xgb'])
    mean_n = np.mean(df['y_naive'])
    
    # Calculate mean accuracy for XGB and Naive models
    mean_accuracy = (1 - (np.abs(mean_a - mean_p) / mean_a)) * 100
    accuracy = np.mean(df['acc_xgb'])
    
    mean_accuracy_n = (1 - (np.abs(mean_a - mean_n) / mean_a)) * 100
    accuracy_n = np.mean(df['acc_naive'])
    
    # Calculate standard deviations of accuracy
    sd_acc = np.std(df['acc_xgb'])
    sd_acc_n = np.std(df['acc_naive'])
    
    # Calculate bias and positive/negative bias for XGB and Naive models
    bias = np.mean(df['bias_xgb'])
    positive_bias = np.mean(df['bias_xgb'][df['bias_xgb'] > 0])
    negative_bias = np.mean(df['bias_xgb'][df['bias_xgb'] < 0])
    
    bias_n = np.mean(df['bias_naive'])
    positive_bias_n = np.mean(df['bias_naive'][df['bias_naive'] > 0])
    negative_bias_n = np.mean(df['bias_naive'][df['bias_naive'] < 0])

    # Calculate sum of adjusted bias
    adj_bias = np.sum(df['adj_bias_xgb'])
    adj_bias_n = np.sum(df['adj_bias_naive'])
    diff_xgb_n = adj_bias_n - adj_bias
    
    # Calculate standard deviations of bias
    sd_bias = np.std(df['bias_xgb'])
    sd_bias_n = np.std(df['bias_naive'])
    
    # Store results in a dictionary
    results = {
        'Metric': [
            'Mean Accuracy', 'Accuracy', 'Mean Accuracy Naive', 'Accuracy Naive',
            'Standard Deviation of Accuracy', 'Standard Deviation of Accuracy Naive',
            'Bias', 'Positive Bias', 'Negative Bias', 
            'Bias Naive', 'Positive Bias Naive', 'Negative Bias Naive',
            'Standard Deviation of Bias', 'Standard Deviation of Bias Naive',
            'Money XGB', 'Money Naive', 'Difference naive - XGB'
        ],
        'Value': [
            mean_accuracy, accuracy, mean_accuracy_n, accuracy_n,
            sd_acc, sd_acc_n,
            bias, positive_bias, negative_bias,
            bias_n, positive_bias_n, negative_bias_n,
            sd_bias, sd_bias_n,
            adj_bias, adj_bias_n, diff_xgb_n
        ]
    }
    
    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

In [ ]:
def calculate_metrics_grouped(df, column):
    store_stats = []
    
    # Iterate over each group's data
    for store, group in df.groupby(column):
        # # Skip calculation if the group value is 0 or missing (NaN)
        # if store == 0 or pd.isna(store):
        #     continue
        
        # Calculate statistics for the current group using the existing function
        df_metrics = calculate_metrics(group)
        
        # Add the column value (e.g., store number) to each metric's result for reference
        df_metrics[column] = store
        
        # Append each group's result to the list
        store_stats.append(df_metrics)
    
    # Concatenate all group statistics into a single DataFrame
    store_means = pd.concat(store_stats).reset_index(drop=True)

    # Pivot the DataFrame to make column values into columns and Metric rows
    store_metrics_pivot = store_means.pivot(index='Metric', columns=column, values='Value')

    # Drop columns with all NA values
    store_metrics_pivot = store_metrics_pivot.dropna(axis=1, how='all')

    return store_metrics_pivot

In [ ]:
def summarize_store_data(df, group_column, summary_columns, operation='sum'):
    if operation == 'sum':
        grouped_df = df.groupby(group_column)[summary_columns].sum().reset_index()
    elif operation == 'mean':
        grouped_df = df.groupby(group_column)[summary_columns].mean().reset_index()
    else:
        raise ValueError("Invalid operation. Choose 'sum' or 'mean'.")
    
    return grouped_df

In [ ]:
def sorted_histogram(dict_df, group_name):
    """
    Plots sorted histograms for a specific group across multiple DataFrames from a dictionary.

    Parameters:
        dict_df (dict): A dictionary where keys are group names and values are DataFrames.
        group_name (str): The name of the row to plot.

    Returns:
        None
    """
    for key, df in dict_df.items():
        print(f"Processing group: {key}")
        
        # Ensure the row exists in the DataFrame
        if group_name not in df.index:
            raise ValueError(f"Row '{group_name}' not found in the DataFrame index of group '{key}'.")
        
        # Extract the specified row and convert to a Series
        difference_row = df.loc[group_name]
        
        # Sort the values by descending order
        difference_sorted = difference_row.sort_values(ascending=False)
        
        # Plot the histogram
        plt.figure(figsize=(10, 6))
        plt.bar(difference_sorted.index.astype(str), difference_sorted, color='skyblue', edgecolor='black')
        plt.title(f'Histogram of {group_name} (Sorted) for Group: {key}', fontsize=14)
        plt.xlabel(key, fontsize=12)
        plt.ylabel(f'{group_name}', fontsize=12)
        plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()


In [ ]:
def plot_bias_comparison(df, group_name):
    """
    Plots a comparison of Negative Bias, Negative Bias Naive, Positive Bias, and Positive Bias Naive
    across the specified group from a given DataFrame, sorted by Positive Bias Naive - Positive Bias.

    Parameters:
    df (pd.DataFrame): DataFrame where rows represent metrics ('Negative Bias', 'Negative Bias Naive',
                        'Positive Bias', 'Positive Bias Naive') and columns are group values (e.g., store numbers).
    group_name (str): The name of the group being plotted (e.g., 'store_type', 'store_cluster').
    """
    # Extract the data to plot
    group_values = list(df.columns)  # Get group values (e.g., stores, clusters) as list of columns
    negative_bias = df.loc['Negative Bias', group_values].values.astype(float)  # Negative Bias values
    negative_bias_naive = df.loc['Negative Bias Naive', group_values].values.astype(float)  # Negative Bias Naive values
    positive_bias = df.loc['Positive Bias', group_values].values.astype(float)  # Positive Bias values
    positive_bias_naive = df.loc['Positive Bias Naive', group_values].values.astype(float)  # Positive Bias Naive values

    # Calculate the difference for sorting: Positive Bias Naive - Positive Bias
    bias_diff = positive_bias_naive - positive_bias

    # Sort the group values based on the difference in descending order
    sorted_indices = np.argsort(bias_diff)[::-1]  # Sort indices in descending order
    sorted_group_values = np.array(group_values)[sorted_indices]  # Reorder group values based on sorted indices

    # Reorder all other data according to sorted group values
    negative_bias = negative_bias[sorted_indices]
    negative_bias_naive = negative_bias_naive[sorted_indices]
    positive_bias = positive_bias[sorted_indices]
    positive_bias_naive = positive_bias_naive[sorted_indices]

    # Set up the plot
    x = np.arange(len(sorted_group_values))  # Group positions for the x-axis
    width = 0.2  # Width of the bars

    fig, ax = plt.subplots(figsize=(15, 8))

    # Plot the bars for each category
    ax.bar(x - width, negative_bias, width, label='Negative Bias')
    ax.bar(x, negative_bias_naive, width, label='Negative Bias Naive')
    ax.bar(x + width, positive_bias, width, label='Positive Bias')
    ax.bar(x + 2 * width, positive_bias_naive, width, label='Positive Bias Naive')

    # Add labels and formatting
    ax.set_xlabel(group_name.capitalize())  # Use the group name for the x-axis label
    ax.set_ylabel('Bias Value')
    ax.set_title(f'Comparison of Negative and Positive Bias Across {group_name.capitalize()}')
    ax.set_xticks(x)
    ax.set_xticklabels(sorted_group_values, rotation=90)
    ax.legend()

    # Display the plot
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_sales_comparison(df_pred, filter_column, filter_value):
    """
    Plots a comparison of predicted sales (y_xgb), actual sales (unit_sales), and naive predictions (y_naive)
    over weeks for a specified store (or any other column-based filter).

    Parameters:
    df_pred (pd.DataFrame): DataFrame containing the data.
    filter_column (str): The column name to filter by (e.g., 'store_nbr').
    filter_value (int or str): The value to filter the specified column by (e.g., a store number).
    """
    # Step 1: Filter data for the specified column and value
    grouped_df = df_pred[df_pred[filter_column] == filter_value]

    # Step 2: Group by 'week_number_cum' and compute the sum for each of the columns
    grouped_df = grouped_df.groupby('week_number_cum')[['y_xgb', 'unit_sales', 'y_naive']].sum().reset_index()

    # Step 3: Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='Predicted Sales (y_xgb)', color='blue')
    plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction (y_naive)', color='red')

    # Add titles and labels
    plt.title(f'Sum of y_xgb and Unit Sales Over Weeks for {filter_column} = {filter_value}')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Sales')
    plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()


Dataframe with overall metrics

In [ ]:
pd.set_option('display.float_format', '{:.6f}'.format)
df_metrics = calculate_metrics(df_pred)
df_metrics

Dataframes with metrics per storetypes, storeclusters, stores, perishable, itemfamily, itemclass, weeknumbercum

In [ ]:
list_grouping = ['store_type', 'store_cluster', 'store_nbr', 'perishable', 'item_family', 'item_class', 'week_number_cum']

# Initialize an empty dictionary
dict_df_metrics = {}

for i in list_grouping:
    # Calculate metrics for the current grouping
    df_metrics_grouped = calculate_metrics_grouped(df_pred, i)
    dict_df_metrics[i] = df_metrics_grouped  # Add the resulting DataFrame to the dictionary with `i` as the key
    print(df_metrics_grouped)

Dataframes with sum of actual sales, naive sales, predicted sales, accuracy, bias and adjusted bias (money)

In [ ]:
for i in list_grouping:
    df_sum = summarize_store_data(df_pred, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'acc_xgb', 'acc_naive', 'adj_bias_xgb', 'adj_bias_naive'],
                              operation='sum')
    print(df_sum)
    

Dataframes with mean of actual sales, naive sales, predicted sales, accuracy, bias and adjusted bias (money)


In [ ]:
for i in list_grouping:
    df_mean = summarize_store_data(df_pred, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'acc_xgb', 'acc_naive', 'adj_bias_xgb', 'adj_bias_naive'],
                              operation='mean')
    print(df_mean)

Histograms with differences between naive predictions and xgb predictions (sorted largest to smallest)

In [ ]:
sorted_histogram(dict_df_metrics, 'Difference naive - XGB')

Histograms with positive and negative bias naive vs xgb, per store, per item family etc.

In [ ]:
# for group_name, df in dict_df_metrics.items():
#     print(f"Plotting for group: {group_name}")
#     plot_bias_comparison(df, group_name)
#     plot_bias_comparison

Linediagram to show actual sales, predicted sales by xgb vs naive

In [ ]:
# Choose from list_grouping a column, get unique values, and create line graphs with total actual sales, predicted sales and naive prediction

column = 'item_family'
# Get unique values in the current column
unique_values = df_pred[column].unique()
    
# Loop through each unique value in the column
for value in unique_values:
    # Call the plot_sales_comparison function
    plot_sales_comparison(df_pred, filter_column=column, filter_value=value)


In [ ]:
#grouped_df = df_pred.groupby('week_number_cum')[['y_pred', 'unit_sales']].sum().reset_index()
grouped_df = df_pred[(df_pred['item_nbr'] == 103520) & (df_pred['store_nbr'] == 1)]

# Step 2: Plot the results
plt.figure(figsize=(12, 6))
plt.plot(grouped_df['week_number_cum'], grouped_df['bias_xgb'], marker='o', label='Predicted Sales (y_pred)', color='blue')
# plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')
plt.plot(grouped_df['week_number_cum'], grouped_df['bias_naive'], marker='o', label='Naive prediction', color='red')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()

In [ ]:
df = calculate_metrics_grouped(df_pred, 'week_number_cum')

# Assuming your DataFrame is named 'df'

# Extract the row for "Difference naive - XGB" and transpose it to work with it as a Series
difference_row = df.loc['Difference naive - XGB'].transpose()

# Ensure the index represents week numbers and the values are numeric
week_numbers = df.columns.astype(int)  # Assuming column names are week numbers
difference_values = pd.to_numeric(difference_row, errors='coerce')

# Compute the cumulative sum of the difference values
cumulative_difference = difference_values.cumsum()

# Plot the cumulative difference
plt.figure(figsize=(12, 6))
plt.plot(week_numbers, cumulative_difference, marker='o', label='Cumulative Difference')
plt.title('Cumulative Difference Naive - XGB Over Time', fontsize=14)
plt.xlabel('Week Number', fontsize=12)
plt.ylabel('Cumulative Difference', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


